# Reinforcement Learning: DQN and Policy Gradients

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Understand **reinforcement learning** (agent, environment, reward, policy)
- Run a **simple environment** (e.g. CartPole from Gym) and see random vs a few steps of learning
- See the idea of **Q-learning** / **policy gradients** (we use RL instead of supervised learning when we have rewards, not labels)

---

## 🌍 Real life

**Where is this used?** RL is used in **games**, **robotics**, **recommendation**, and **autonomous driving** when we have **rewards** (or penalties) instead of labeled data.

**In this notebook we use** an **RL environment** (e.g. CartPole) and take **random actions** (or a simple policy) to see how reward accumulates. We use **reinforcement learning** (instead of supervised learning) **because** we don't have "correct" actions; we have **rewards** and the agent learns by trial and error.

**📌 Covers slide(s):** **18** — Reinforcement Learning (DQN, policy gradients). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell. Install **gymnasium** if needed: `pip install gymnasium` (required; old `gym` is not used here).

## Theory (short)

- **RL:** Agent takes **actions** in an **environment**; gets **rewards** (or penalties). Goal: maximize cumulative reward. No labeled (state, action) pairs; we learn from experience.
- **DQN:** Deep Q-Network; a neural network approximates Q(s, a) (expected return for state s, action a). We use **DQN** (instead of tabular Q-learning) when the state space is large or continuous.
- **Policy gradients:** Directly optimize a **policy** (probability of actions) using gradient ascent on expected return. Used in actor-critic, PPO, etc.
- **We use an environment** (e.g. CartPole) so you see state, action, reward in a few steps without full DQN training (which takes longer).

## 📥 Inputs & 📤 Outputs

**Inputs:** NumPy, optional `gym` or `gymnasium`. We use **CartPole** (balance a pole on a cart) so the notebook runs without extra setup.

**Dataset:** Real — CartPole (OpenAI Gym environment).

**Outputs:** Printed steps: state, action, reward, done; total reward over one or a few episodes. Optional: simple plot of reward per step.

## Step 1: Imports and create environment (we use Gym/Gymnasium to get state–action–reward interface)

In [1]:
import numpy as np

try:
    import gymnasium as gym
    ENV_NAME = "CartPole-v1"
    HAS_GYM = True
except ImportError:
    HAS_GYM = False

if HAS_GYM:
    env = gym.make(ENV_NAME)
    print("Environment:", ENV_NAME)
    print("Action space:", env.action_space)
    print("Observation space:", env.observation_space)
else:
    print("Install gym: pip install gymnasium (or pip install gym)")

Install gym: pip install gymnasium (or pip install gym)


## Step 2: Run one episode with random actions (we use random policy to show how reward is collected; DQN would learn a better policy)

In [2]:
if HAS_GYM:
    reset_result = env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    total_reward = 0
    for step in range(100):
        action = env.action_space.sample()
        result = env.step(action)
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        total_reward += reward
        if done:
            break
    env.close()
    print("Steps:", step + 1, "Total reward:", total_reward)
    print("RL: agent learns to maximize total reward; DQN/policy gradients train a neural network policy.")

*(Optional)* For full DQN: add a Q-network, replay buffer, and train for many episodes.

In [3]:
# (No extra code; Step 2 above runs one episode. Uncomment and run again to see variance in total reward.)

## 🌍 Real-World Worked Example — REINFORCE on CartPole

**Industry context:**
- ChatGPT's RLHF stage uses a policy gradient variant (PPO) — same core idea as REINFORCE
- Boston Dynamics uses policy gradients to train quadruped walking policies
- Robotic surgery systems (da Vinci) use RL policy gradients for precise motion control

We implement **REINFORCE** (the simplest policy gradient) on CartPole-v1.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)
env = gym.make('CartPole-v1')

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 2), nn.Softmax(dim=-1)
        )
    def forward(self, x): return self.net(x)

policy  = PolicyNet()
opt     = optim.Adam(policy.parameters(), lr=2e-3)
GAMMA   = 0.99; ep_rewards = []

for episode in range(500):
    obs,_ = env.reset(); log_probs=[]; rewards_ep=[]
    for _ in range(500):
        probs  = policy(torch.tensor(obs).float().unsqueeze(0))
        dist   = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        obs, r, done, trunc, _ = env.step(action.item())
        rewards_ep.append(r)
        if done or trunc: break

    # ── Compute discounted returns ──────────────────────────────────────────
    G = 0; returns = []
    for r in reversed(rewards_ep):
        G = r + GAMMA*G; returns.insert(0, G)
    returns = torch.tensor(returns).float()
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    # ── Policy gradient loss ────────────────────────────────────────────────
    loss = -torch.stack([lp*R for lp,R in zip(log_probs, returns)]).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    ep_rewards.append(sum(rewards_ep))
    if episode%50==0: print(f"Episode {episode:3d} | Avg reward: {np.mean(ep_rewards[-20:]):.1f}")

env.close()
ma = np.convolve(ep_rewards, np.ones(30)/30, 'valid')
plt.plot(ep_rewards, alpha=0.3, label="Episode reward")
plt.plot(ma, lw=2, label="30-ep average")
plt.axhline(475, color='red', linestyle='--', label="Solved")
plt.title("REINFORCE on CartPole (same algorithm behind ChatGPT RLHF)")
plt.xlabel("Episode"); plt.ylabel("Reward"); plt.legend(); plt.tight_layout(); plt.show()

## 🧩 Mini-exercise

**Try it:** Run 20 episodes with the random agent and store the total reward per episode. Print the mean and max reward. Then try a different environment (e.g. `LunarLander-v2` if installed) and compare.

---

## ✅ Summary

**What you did:** Ran a simple RL environment (CartPole) with random actions and saw total reward. This illustrates the RL loop: state → action → reward.

**In real life you'd also:** Implement DQN (Q-network + replay buffer) or policy gradients (e.g. REINFORCE), train for many episodes, and tune hyperparameters.

**The main idea:** Reinforcement learning learns from **rewards** (not labels); DQN and policy gradients are two ways to learn a good policy with neural networks.

**Next:** `01_gans_and_autoencoders_vaes` covers GANs/VAEs; for full DQN see dedicated RL tutorials.

## 📚 References & Further Reading

**Papers:**
- Williams (1992) — [REINFORCE](https://link.springer.com/article/10.1007/BF00992696)
- Schulman et al. (2017) — [PPO: Proximal Policy Optimization](https://arxiv.org/abs/1707.06347)
- Mnih et al. (2016) — [A3C: Asynchronous Methods for Deep RL](https://arxiv.org/abs/1602.01783)

**State-of-the-Art:**
- OpenAI's ChatGPT uses RLHF (PPO) to align responses with human preferences
- Boston Dynamics robots use PPO for locomotion control